In [3]:
pip install -q transformers datasets evaluate torch langchain langchain-core langchain-community faiss-cpu langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 85.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 66.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have

# Dataset Formation and making the corpus

In [7]:
from datasets import load_dataset
data = load_dataset('hotpotqa/hotpot_qa','distractor')

In [8]:
data['train']

Dataset({
    features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
    num_rows: 90447
})

In [ ]:
# data['train'][0]

In [9]:
data['train'][0]['context']['title']

['Radio City (Indian radio station)',
 'History of Albanian football',
 'Echosmith',
 "Women's colleges in the Southern United States",
 'First Arthur County Courthouse and Jail',
 "Arthur's Magazine",
 '2014–15 Ukrainian Hockey Championship',
 'First for Women',
 'Freeway Complex Fire',
 'William Rast']

In [10]:
data

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 90447
    })
    validation: Dataset({
        features: ['id', 'question', 'answer', 'type', 'level', 'supporting_facts', 'context'],
        num_rows: 7405
    })
})

In [6]:
corpus = {}
for split in ["train", "validation"]:
    for datapoint in data[split]:
        titles  = datapoint['context']['title']
        sentences_list = datapoint['context']['sentences']
        for title,sentences_list in zip(titles,sentences_list):
            text = " ".join(sentences_list) 
            corpus[title] = {
                "title": title,
                "text": text
            }
    

In [7]:
len(corpus)

507494

In [8]:
print(next(iter(corpus)))

Radio City (Indian radio station)


In [9]:
print(data["train"][0]["context"]["title"][:5])
print(data["train"][1]["context"]["title"][:5])
print(data["train"][2]["context"]["title"][:5])

['Radio City (Indian radio station)', 'History of Albanian football', 'Echosmith', "Women's colleges in the Southern United States", 'First Arthur County Courthouse and Jail']
['Ritz-Carlton Jakarta', 'Oberoi family', 'Ishqbaaaz', 'Hotel Tallcorn', 'Mohan Singh Oberoi']
['Lisa Simpson', 'Marge Simpson', 'Bart Simpson', 'Allie Goertz', 'Milhouse Van Houten']


In [10]:
print(len(set(corpus.keys())))
print(len(corpus))

507494
507494


# Chunking

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [13]:
lengths = []

for article in corpus.values():
    text = " ".join(article["text"].split())
    token_count = len(tokenizer.tokenize(text))
    lengths.append(token_count)

In [14]:
print(f"Number of articles : {len(lengths)}")
print(f"Minimum tokens     : {min(lengths)}")
print(f"Maximum tokens     : {max(lengths)}")
print(f"Average tokens     : {sum(lengths)/len(lengths):.2f}")

Number of articles : 507494
Minimum tokens     : 5
Maximum tokens     : 1786
Average tokens     : 101.65


In [15]:
thresholds = [128, 256, 384, 512]

for t in thresholds:
    count = sum(length > t for length in lengths)
    print(f">{t} tokens : {count} ({count/len(lengths)*100:.2f}%)")

>128 tokens : 128630 (25.35%)
>256 tokens : 8482 (1.67%)
>384 tokens : 924 (0.18%)
>512 tokens : 182 (0.04%)


In [16]:
# converting the corpus into documents
from langchain_core.documents import Document
documents = []

for article in corpus.values():
    text = " ".join(article["text"].split())

    documents.append(
        Document(
            page_content=f"Title: {article['title']}\n\nContent:\n{text}",
            metadata={
                "title": article["title"]
            }
        )
    )

In [17]:
print(documents[0].page_content)
print("=" * 80)
print(documents[1].page_content)
print("=" * 80)
print(documents[2].page_content)

Title: Radio City (Indian radio station)

Content:
Radio City is India's first private FM radio station and was started on 3 July 2001. It broadcasts on 91.1 (earlier 91.0 in most cities) megahertz from Mumbai (where it was started in 2004), Bengaluru (started first in 2001), Lucknow and New Delhi (since 2003). It plays Hindi, English and regional songs. It was launched in Hyderabad in March 2006, in Chennai on 7 July 2006 and in Visakhapatnam October 2007. Radio City recently forayed into New Media in May 2008 with the launch of a music portal - PlanetRadiocity.com that offers music related news, videos, songs, and other music-related features. The Radio station currently plays a mix of Hindi and Regional music. Abraham Thomas is the CEO of the company.
Title: History of Albanian football

Content:
Football in Albania existed before the Albanian Football Federation (FSHF) was created. This was evidenced by the team's registration at the Balkan Cup tournament during 1929-1931, which st

In [24]:
len(documents)

507494

# Embedding Model and embedding the documents

In [11]:
from langchain_huggingface import HuggingFaceEmbeddings

In [12]:
embedding = HuggingFaceEmbeddings(
    model_name= 'BAAI/bge-base-en-v1.5'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [13]:
from langchain_community.vectorstores import FAISS

/tmp/ipykernel_58/1059762981.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [25]:
batch_size = 1000

vector_store = FAISS.from_documents(
    documents[:batch_size],
    embedding = embedding
)

In [26]:
next_milestone = 100000

for i in range(batch_size, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    vector_store.add_documents(batch)

    indexed = min(i + batch_size, len(documents))
    while indexed >= next_milestone:
        print(f"✅ Indexed {next_milestone:,} documents")
        next_milestone += 100000

print(f"Finished indexing all {len(documents):,} documents")

✅ Indexed 100,000 documents
✅ Indexed 200,000 documents
✅ Indexed 300,000 documents
✅ Indexed 400,000 documents
✅ Indexed 500,000 documents
Finished indexing all 507,494 documents


In [27]:
vector_store.save_local("hotpotqa_faiss")

In [17]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    print(root)
    for f in files:
        print("   ", f)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/ommtripathi
/kaggle/input/datasets/ommtripathi/hotpotqa
    index.faiss
    index.pkl


In [14]:
vector_store = FAISS.load_local(
    "/kaggle/input/datasets/ommtripathi/hotpot",
    embedding,
    allow_dangerous_deserialization=True
)

In [15]:
retriever = vector_store.as_retriever(
    search_type = "similarity",
    search_kwargs={
    "k": 10 ,
    } 
)


In [16]:
docs = retriever.invoke("Which magazine was started first Arthur's Magazine or First for Women?")
docs

[Document(id='83cb0ea9-83ed-4321-9c36-e90fbedf5ac0', metadata={'title': "Arthur's Lady's Home Magazine"}, page_content='Title: Arthur\'s Lady\'s Home Magazine\n\nContent:\nArthur\'s Home Magazine (1852-ca.1898) or Ladies\' Home Magazine was an American periodical published in Philadelphia by Timothy Shay Arthur. Editors Arthur and Virginia Francis Townsend selected writing and illustrations intended to appeal to female readers. Among the contributors: Mary Tyler Peabody Mann and Kate Sutherland. In its early years the monthly comprised a selection of articles originally published in Arthur\'s weekly "Home Gazette." Its nonfiction stories contained occasional factual inaccuracies for the sake of a good read. A contemporary review judged it "gotten up in good taste and well; and is in nothing overdone. Even its fashion plates are not quite such extravagant caricatures of rag-baby work as are usually met with in some of the more fancy magazines." Readers included patrons of the Mercantile

In [21]:
data['train'][0]["supporting_facts"]["title"]

["Arthur's Magazine", 'First for Women']

Function to check how accurate the answers are getting retrieved

In [31]:
def evaluate_retriever(example, retriever):
    question = example["question"]

    gold_titles = set(example["supporting_facts"]["title"])

    retrieved_docs = retriever.invoke(question)

    retrieved_titles = {
        doc.metadata["title"]
        for doc in retrieved_docs
    }

    correct = len(gold_titles & retrieved_titles)
    recall = correct / len(gold_titles)

    return recall

In [32]:
evaluate_retriever(data["train"][0], retriever)

1.0

In [35]:
# score for the first 100 documents
scores = []


for k in [3, 5, 10, 20]:
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={
            "k": k,
            # "fetch_k": max(20, 2 * k),
            
        }
    )

    scores = []

    for example in data["train"].select(range(100)):
        scores.append(evaluate_retriever(example, retriever))

    print(f"Recall@{k}: {sum(scores)/len(scores):.4f}")

Recall@3: 0.7100
Recall@5: 0.7650
Recall@10: 0.8150
Recall@20: 0.8200


# Building the Reader Model - FineTuning De-Bert

In [14]:
def build_context(example, retriever):
    question = example["question"]

    gold_titles = set(example["supporting_facts"]["title"])

    retrieved_docs = retriever.invoke(question)

    retrieved_supporting_docs = [
        doc
        for doc in retrieved_docs
        if doc.metadata["title"] in gold_titles
    ]

    return retrieved_supporting_docs, gold_titles

In [16]:
def prepare_train_example(example, retriever):
    question = example["question"]
    answer = example["answer"]

    # Skip yes/no for now
    if answer.lower() in ["yes", "no"]:
        return None

    retrieved_supporting_docs, gold_titles = build_context(
        example,
        retriever
    )

    # If retriever missed some supporting docs,
    # use the gold ones from the dataset
    if len(retrieved_supporting_docs) < len(gold_titles):

        title_to_text = {}

        titles = example["context"]["title"]
        sentences = example["context"]["sentences"]

        for title, sent_list in zip(titles, sentences):
            title_to_text[title] = (
                f"Title: {title}\n\n"
                f"Content:\n{' '.join(sent_list)}"
            )

        context = "\n\n".join(
            title_to_text[title]
            for title in gold_titles
        )

        retrieval_success = False

    else:

        context = "\n\n".join(
            doc.page_content
            for doc in retrieved_supporting_docs
        )

        retrieval_success = True

    answer_start = context.find(answer)

    if answer_start == -1:
        return None

    return {
        "id": example["id"],
        "question": question,
        "context": context,
        "answers": {
            "text": [answer],
            "answer_start": [answer_start]
        },
        "retrieval_success": retrieval_success
    }

In [17]:
def prepare_eval_example(example, retriever):
    question = example["question"]
    answer = example["answer"]

    if answer.lower() in ["yes", "no"]:
        return None

    retrieved_supporting_docs, gold_titles = build_context(
        example,
        retriever
    )

    context = "\n\n".join(
        doc.page_content
        for doc in retrieved_supporting_docs
    )

    answer_start = context.find(answer)

    # Retriever failed
    if answer_start == -1:
        return None

    return {
        "id": example["id"],
        "question": question,
        "context": context,
        "answers": {
            "text": [answer],
            "answer_start": [answer_start]
        }
    }

In [18]:
from tqdm.auto import tqdm
from datasets import Dataset

In [19]:
train_reader = []

for example in tqdm(data["train"]):
    sample = prepare_train_example(example, retriever)

    if sample is not None:
        train_reader.append(sample)

train_reader = Dataset.from_list(train_reader)

  0%|          | 0/90447 [00:00<?, ?it/s]

In [22]:
validation_reader = []

for example in tqdm(data["validation"]):
    sample = prepare_eval_example(example, retriever)

    if sample is not None:
        validation_reader.append(sample)

validation_reader = Dataset.from_list(validation_reader)

  0%|          | 0/7405 [00:00<?, ?it/s]

In [20]:
train_reader.save_to_disk("reader_train")

Saving the dataset (0/1 shards):   0%|          | 0/84959 [00:00<?, ? examples/s]

In [21]:
import shutil

shutil.make_archive(
    "/kaggle/working/reader_train",
    "zip",
    "/kaggle/working/reader_train"
)

'/kaggle/working/reader_train.zip'

In [23]:
validation_reader.save_to_disk("reader_validation")

Saving the dataset (0/1 shards):   0%|          | 0/4687 [00:00<?, ? examples/s]

In [24]:
import shutil

shutil.make_archive(
    "/kaggle/working/reader_validation",
    "zip",
    "/kaggle/working/reader_validation"
)

'/kaggle/working/reader_validation.zip'

# Saving the datasets

In [25]:
!pip install -q huggingface_hub

In [31]:
from huggingface_hub import login
login()

In [34]:
# from datasets import load_from_disk

# dataset = load_from_disk("reader_train")

# dataset.push_to_hub(
#     "ommtripathi/hotpotqa-reader-train"
# )

# Model Training

In [23]:
!cp -r /kaggle/input/datasets/ommtripathi/hotpot-data/reader_train /kaggle/working/
!cp -r /kaggle/input/datasets/ommtripathi/hotpot-data/reader_validation /kaggle/working/

In [24]:
from datasets import load_from_disk


train_dataset = load_from_disk("/kaggle/working/reader_train")
validation_dataset = load_from_disk("/kaggle/working/reader_validation")

print(train_dataset)
print(validation_dataset)

Dataset({
    features: ['id', 'question', 'context', 'answers', 'retrieval_success'],
    num_rows: 84959
})
Dataset({
    features: ['id', 'question', 'context', 'answers'],
    num_rows: 4687
})


In [19]:
from transformers import AutoTokenizer

checkpoint = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [20]:
max_length = 512
doc_stride = 128

In [25]:
def preprocess_function(examples):
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        max_length=max_length,
        truncation="only_second",
        stride=doc_stride,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):

        sample_idx = sample_mapping[i]
        answer = examples["answers"][sample_idx]

        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])

        sequence_ids = tokenized_examples.sequence_ids(i)

        # Find where the context starts
        token_start_index = 0
        while sequence_ids[token_start_index] != 1:
            token_start_index += 1

        # Find where the context ends
        token_end_index = len(sequence_ids) - 1
        while sequence_ids[token_end_index] != 1:
            token_end_index -= 1

        # If answer not inside this span
        if (
            offsets[token_start_index][0] > start_char
            or offsets[token_end_index][1] < end_char
        ):
            start_positions.append(0)
            end_positions.append(0)
            continue

        # Move token_start_index
        while (
            token_start_index < len(offsets)
            and offsets[token_start_index][0] <= start_char
        ):
            token_start_index += 1

        start_positions.append(token_start_index - 1)

        # Move token_end_index
        while offsets[token_end_index][1] >= end_char:
            token_end_index -= 1

        end_positions.append(token_end_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions

    return tokenized_examples

In [26]:
train_tokenized = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names,
)

validation_tokenized = validation_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=validation_dataset.column_names,
)

Map:   0%|          | 0/84959 [00:00<?, ? examples/s]

Map:   0%|          | 0/4687 [00:00<?, ? examples/s]

In [27]:
print(train_tokenized)
print(train_tokenized[0])

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions'],
    num_rows: 85120
})
{'input_ids': [2597, 3421, 284, 696, 362, 7674, 280, 268, 5593, 289, 1244, 270, 2568, 302, 7181, 294, 7674, 280, 268, 5593, 5825, 294, 7674, 280, 268, 5593, 287, 105393, 958, 101737, 285, 284, 299, 733, 7170, 50747, 1378, 267, 5080, 267, 262, 1316, 474, 1880, 260, 23545, 293, 897, 260, 430, 260, 7674, 261, 278, 3103, 374, 293, 17520, 336, 260, 25119, 261, 851, 260, 1773, 260, 80483, 261, 4537, 4060, 452, 19424, 261, 2651, 1098, 260, 44321, 261, 263, 690, 260, 344, 903, 42106, 278, 284, 15085, 352, 307, 11335, 4916, 280, 268, 5087, 280, 268, 2538, 309, 260, 7181, 294, 1244, 270, 2568, 5825, 294, 1244, 270, 2568, 269, 266, 1220, 280, 268, 3421, 1378, 293, 25562, 2916, 1543, 267, 262, 2222, 260, 279, 3421, 284, 696, 267, 6507, 260, 325, 269, 636, 267, 54498, 46241, 261, 485, 3744, 260, 344, 1513, 262, 8806, 265, 262, 3421, 284, 376, 261, 22397, 261, 47964, 

In [28]:
from transformers import AutoModelForQuestionAnswering

model = AutoModelForQuestionAnswering.from_pretrained(
    "microsoft/deberta-v3-base"
)

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
qa_outputs.bias                         | MISSING    | 
qa_outputs.weight                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

In [29]:
from transformers import DefaultDataCollator

data_collator = DefaultDataCollator()

In [40]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./deberta-hotpotqa",

    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    dataloader_num_workers=8,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_steps=100,

    fp16=False,
    bf16=False,

    load_best_model_at_end=True,

    report_to="none"
)

In [32]:
import transformers
print(transformers.__version__)

5.0.0


In [63]:
example = train_tokenized[0]

start = example["start_positions"]
end = example["end_positions"]

print(start, end)

print(tokenizer.decode(example["input_ids"][start:end+1]))

16 19
Arthur's Magazine


In [64]:
bad = 0

for ex in train_tokenized:
    if ex["start_positions"] > ex["end_positions"]:
        bad += 1

print(bad)

4


In [65]:
print(train_tokenized.features)

{'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8')), 'start_positions': Value('int64'), 'end_positions': Value('int64')}


In [34]:
import torch
print(torch.get_default_dtype())
model = AutoModelForQuestionAnswering.from_pretrained(
    "microsoft/deberta-v3-base"
).float()

print(next(model.parameters()).dtype)

torch.float32


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForQuestionAnswering LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
qa_outputs.bias                         | MISSING    | 
qa_outputs.weight                       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

torch.float32


In [41]:


model.eval()

batch = train_tokenized[:2]
batch = {k: torch.tensor(v).to(model.device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(**batch)

print(outputs.loss)
print(torch.isnan(outputs.start_logits).any())
print(torch.isnan(outputs.end_logits).any())

tensor(5.6472, device='cuda:0')
tensor(False, device='cuda:0')
tensor(False, device='cuda:0')


In [42]:
import torch

print("CUDA devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

CUDA devices: 2
0 Tesla T4
1 Tesla T4


In [84]:
print(trainer.args.n_gpu)

2


In [43]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,
    data_collator=data_collator,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


OutOfMemoryError: Caught OutOfMemoryError in replica 0 on device 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/parallel_apply.py", line 103, in _worker
    output = module(*input, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/models/deberta_v2/modeling_deberta_v2.py", line 1216, in forward
    outputs = self.deberta(
              ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*arg

In [1]:
trainer.save_model("deberta-hotpotqa")
tokenizer.save_pretrained("deberta-hotpotqa")

NameError: name 'trainer' is not defined